Silver reproducibility notebook

- Configure `DATA_DOI`, `MODEL_DOI`, filenames, and switches in the next cell.
- Loads data from local `Data/` or downloads from Zenodo, verifies optional hashes, sets deterministic seeds, trains or loads a PyTorch model, captures environment.

In [1]:
# Configuration and reproducibility helpers
from __future__ import annotations
import os, sys, json, hashlib, random, pathlib
from typing import Optional
import numpy as np

# Bronze DOIs used by default
DATA_DOI = os.environ.get("DATA_DOI", "10.5281/zenodo.17298664")
MODEL_DOI = os.environ.get("MODEL_DOI", "10.5281/zenodo.17298751")
DATA_FILENAME = os.environ.get("DATA_FILENAME", "simple_dataset.csv")
MODEL_FILENAME = os.environ.get("MODEL_FILENAME", "linear_model.pt")
ARTIFACTS_DIR = os.environ.get("ARTIFACTS_DIR", "artifacts")
DATA_DIR = os.environ.get("DATA_DIR", "Data")
VERBOSE = True
USE_DOWNLOADED_MODEL = os.environ.get("USE_DOWNLOADED_MODEL", "0") == "1"

EXPECTED_DATA_SHA256 = os.environ.get("EXPECTED_DATA_SHA256", "")
EXPECTED_MODEL_SHA256 = os.environ.get("EXPECTED_MODEL_SHA256", "")

GLOBAL_SEED = int(os.environ.get("GLOBAL_SEED", "674"))
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# Utils
def ensure_dir(path: str) -> None:
    pathlib.Path(path).mkdir(parents=True, exist_ok=True)

def sha256_of_file(path: str) -> str:
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha.update(chunk)
    return sha.hexdigest()

def verify_hash(path: str, expected_sha256: str) -> bool:
    if not expected_sha256:
        return True
    actual = sha256_of_file(path)
    if VERBOSE:
        print(f"SHA256 for {path}: {actual}")
    return actual.lower() == expected_sha256.lower()

def zenodo_download(doi: str, filename: str, dest_dir: str) -> str:
    import urllib.request
    ensure_dir(dest_dir)
    record_id = doi.split(".")[-1].replace("zenodo/", "").replace("zenodo-", "").replace("zenodo", "").replace("/", "")
    url = f"https://zenodo.org/records/{record_id}/files/{filename}?download=1"
    dest_path = os.path.join(dest_dir, filename)
    if VERBOSE:
        print(f"Downloading {url} -> {dest_path}")
    urllib.request.urlretrieve(url, dest_path)
    return dest_path

ensure_dir(ARTIFACTS_DIR)
ensure_dir(DATA_DIR)
print("Configuration loaded. Seeds set.")

Configuration loaded. Seeds set.


In [2]:
import os, pandas as pd
# Load dataset from local Data/ or download from Zenodo
local_path = os.path.join(DATA_DIR, DATA_FILENAME)
if not os.path.exists(local_path):
    if not DATA_DOI:
        raise RuntimeError("DATA_DOI is empty and local data not found.")
    local_path = zenodo_download(DATA_DOI, DATA_FILENAME, DATA_DIR)
    if EXPECTED_DATA_SHA256 and not verify_hash(local_path, EXPECTED_DATA_SHA256):
        raise RuntimeError("Downloaded data hash mismatch; refusing to proceed.")
print(f"Using dataset at: {local_path}")
df = pd.read_csv(local_path)
print(df.head())


Using dataset at: Data/simple_dataset.csv
   UserID  Age      City  Salary  Score
0       1   56   Chicago   91717  55.16
1       2   46   Chicago   95859  59.17
2       3   32  New York   71309  51.28
3       4   60   Chicago  108734  71.19
4       5   25   Chicago  115467  54.51


In [3]:
# Preview loaded data
print(df.head())

   UserID  Age      City  Salary  Score
0       1   56   Chicago   91717  55.16
1       2   46   Chicago   95859  59.17
2       3   32  New York   71309  51.28
3       4   60   Chicago  108734  71.19
4       5   25   Chicago  115467  54.51


In [4]:
import os, pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Preprocess
df_processed = pd.get_dummies(df, columns=['City'], drop_first=True)
X = df_processed.drop(['UserID', 'Score'], axis=1).values
y = df_processed['Score'].values.reshape(-1, 1)

# Deterministic split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=GLOBAL_SEED
)

# Torch tensors
X_train_t = torch.tensor(X_train.astype(np.float32))
y_train_t = torch.tensor(y_train.astype(np.float32))
X_test_t = torch.tensor(X_test.astype(np.float32))
y_test_t = torch.tensor(y_test.astype(np.float32))

# Simple linear model
class LinearRegressionModel(nn.Module):
    def __init__(self, input_size: int, output_size: int):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size)
    def forward(self, x):
        return self.linear(x)

model = LinearRegressionModel(X_train.shape[1], 1)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=5e-4)

# Train
num_epochs = 10000
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    preds = model(X_train_t)
    loss = criterion(preds, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    y_pred_t = model(X_test_t)

y_pred = y_pred_t.numpy()
y_true = y_test_t.numpy()

mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
print("--- Model Performance (PyTorch) ---")
print(f"MSE: {mse:.2f}")
print(f"R2: {r2:.2f}")

# Save model artifact
ensure_dir(ARTIFACTS_DIR)
model_path = os.path.join(ARTIFACTS_DIR, MODEL_FILENAME)
torch.save(model.state_dict(), model_path)
print(f"Saved torch state_dict to {model_path}")
print(f"Model SHA256: {sha256_of_file(model_path)}")


Epoch [20/10000], Loss: 105514840.0000
Epoch [40/10000], Loss: 89408448.0000
Epoch [60/10000], Loss: 75170376.0000
Epoch [80/10000], Loss: 62724852.0000
Epoch [100/10000], Loss: 51937228.0000
Epoch [120/10000], Loss: 42661256.0000
Epoch [140/10000], Loss: 34750912.0000
Epoch [160/10000], Loss: 28063396.0000
Epoch [180/10000], Loss: 22460812.0000
Epoch [200/10000], Loss: 17811430.0000
Epoch [220/10000], Loss: 13990893.0000
Epoch [240/10000], Loss: 10883283.0000
Epoch [260/10000], Loss: 8381990.5000
Epoch [280/10000], Loss: 6390317.0000
Epoch [300/10000], Loss: 4821825.5000
Epoch [320/10000], Loss: 3600402.5000
Epoch [340/10000], Loss: 2660037.5000
Epoch [360/10000], Loss: 1944362.7500
Epoch [380/10000], Loss: 1405994.6250
Epoch [400/10000], Loss: 1005727.0000
Epoch [420/10000], Loss: 711616.5000
Epoch [440/10000], Loss: 498045.4062
Epoch [460/10000], Loss: 344781.1875
Epoch [480/10000], Loss: 236088.9219
Epoch [500/10000], Loss: 159913.0625
Epoch [520/10000], Loss: 107153.8594
Epoch [54

Epoch [2780/10000], Loss: 334.7682
Epoch [2800/10000], Loss: 334.7159
Epoch [2820/10000], Loss: 334.6629
Epoch [2840/10000], Loss: 334.6095
Epoch [2860/10000], Loss: 334.5554
Epoch [2880/10000], Loss: 334.5008
Epoch [2900/10000], Loss: 334.4456
Epoch [2920/10000], Loss: 334.3898
Epoch [2940/10000], Loss: 334.3336
Epoch [2960/10000], Loss: 334.2766
Epoch [2980/10000], Loss: 334.2191
Epoch [3000/10000], Loss: 334.1610
Epoch [3020/10000], Loss: 334.1023
Epoch [3040/10000], Loss: 334.0430
Epoch [3060/10000], Loss: 333.9831
Epoch [3080/10000], Loss: 333.9225
Epoch [3100/10000], Loss: 333.8613
Epoch [3120/10000], Loss: 333.7995
Epoch [3140/10000], Loss: 333.7370
Epoch [3160/10000], Loss: 333.6740
Epoch [3180/10000], Loss: 333.6103
Epoch [3200/10000], Loss: 333.5459
Epoch [3220/10000], Loss: 333.4808
Epoch [3240/10000], Loss: 333.4151
Epoch [3260/10000], Loss: 333.3488
Epoch [3280/10000], Loss: 333.2817
Epoch [3300/10000], Loss: 333.2140
Epoch [3320/10000], Loss: 333.1455
Epoch [3340/10000], 

Epoch [5740/10000], Loss: 317.2447
Epoch [5760/10000], Loss: 317.0233
Epoch [5780/10000], Loss: 316.7998
Epoch [5800/10000], Loss: 316.5743
Epoch [5820/10000], Loss: 316.3466
Epoch [5840/10000], Loss: 316.1170
Epoch [5860/10000], Loss: 315.8852
Epoch [5880/10000], Loss: 315.6513
Epoch [5900/10000], Loss: 315.4153
Epoch [5920/10000], Loss: 315.1772
Epoch [5940/10000], Loss: 314.9368
Epoch [5960/10000], Loss: 314.6943
Epoch [5980/10000], Loss: 314.4496
Epoch [6000/10000], Loss: 314.2026
Epoch [6020/10000], Loss: 313.9535
Epoch [6040/10000], Loss: 313.7020
Epoch [6060/10000], Loss: 313.4484
Epoch [6080/10000], Loss: 313.1923
Epoch [6100/10000], Loss: 312.9341
Epoch [6120/10000], Loss: 312.6735
Epoch [6140/10000], Loss: 312.4106
Epoch [6160/10000], Loss: 312.1453
Epoch [6180/10000], Loss: 311.8776
Epoch [6200/10000], Loss: 311.6075
Epoch [6220/10000], Loss: 311.3351
Epoch [6240/10000], Loss: 311.0602
Epoch [6260/10000], Loss: 310.7828
Epoch [6280/10000], Loss: 310.5030
Epoch [6300/10000], 

Epoch [8720/10000], Loss: 252.0419
Epoch [8740/10000], Loss: 251.3228
Epoch [8760/10000], Loss: 250.5995
Epoch [8780/10000], Loss: 249.8718
Epoch [8800/10000], Loss: 249.1399
Epoch [8820/10000], Loss: 248.4038
Epoch [8840/10000], Loss: 247.6634
Epoch [8860/10000], Loss: 246.9189
Epoch [8880/10000], Loss: 246.1701
Epoch [8900/10000], Loss: 245.4170
Epoch [8920/10000], Loss: 244.6599
Epoch [8940/10000], Loss: 243.8985
Epoch [8960/10000], Loss: 243.1329
Epoch [8980/10000], Loss: 242.3632
Epoch [9000/10000], Loss: 241.5893
Epoch [9020/10000], Loss: 240.8113
Epoch [9040/10000], Loss: 240.0291
Epoch [9060/10000], Loss: 239.2429
Epoch [9080/10000], Loss: 238.4526
Epoch [9100/10000], Loss: 237.6582
Epoch [9120/10000], Loss: 236.8597
Epoch [9140/10000], Loss: 236.0572
Epoch [9160/10000], Loss: 235.2507
Epoch [9180/10000], Loss: 234.4402
Epoch [9200/10000], Loss: 233.6257
Epoch [9220/10000], Loss: 232.8073
Epoch [9240/10000], Loss: 231.9849
Epoch [9260/10000], Loss: 231.1586
Epoch [9280/10000], 

In [5]:
# Capture environment for Silver
import subprocess
ensure_dir(ARTIFACTS_DIR)
req_path = os.path.join(ARTIFACTS_DIR, "requirements-silver.txt")
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
with open(req_path, "w") as f:
    f.write(freeze)
print(f"Saved environment to {req_path}")


Saved environment to artifacts/requirements-silver.txt
